# M3 front ends: S1 segmenters, D1 detector and the predictions FE reads

We train and score the candidate front ends, then write the predictions the FE runs read. S1 asks which segmenter
gives the best snake masks and D1 which detector gives the best snake boxes; FE, run in notebook 12, then asks
whether a crop or a mask beats the whole photo. The notebook trains, scores, picks the winners and predicts, in that
order.

**Inputs:** `serpentguard-indian-snakes-clean`, `serpentguard-sam-masks`, `serpentguard-m3-subset` and
`serpentguard-extended-sources` (the Open Images masks and boxes), found with `kagglehub.dataset_download`; the account
running this needs access to all four. The code comes from `../src`, which the runner kernel unpacks into
`/kaggle/working/repo`.

**Outputs:** `/kaggle/working/frontends/` holds each candidate's folder under `S1/` or `D1/` (`run.json`, weights,
`train.log`, `final_metrics.json`), `winners.json`, `S1_table.csv` and `D1_table.csv`.
`/kaggle/working/frontend_preds/` holds `frontend_preds.csv`, `masks_pred/` and `summary.json`; we upload it as the
private dataset `serpentguard-m3-frontend-preds`, which notebook 12 reads for F1, F1m, F2 and F3.

**Run:** on Kaggle with two GPUs and internet on (pip, ImageNet and Ultralytics weights). `SEG_KINDS` and
`TRAIN_DETECTOR` choose what to train, `SEG_EPOCHS` and `DET_EPOCHS` override the epochs, and `LIMIT` makes a smoke
run.

In [1]:
SEG_KINDS = ["maskhead", "lraspp", "unet", "yolo26n-seg"]  # S1 candidates to train; drop one to skip it
TRAIN_DETECTOR = True  # D1(b): YOLO26n fine-tuned on the Grounding DINO boxes
SEG_EPOCHS = None      # None keeps serpentguard.segmenters.SEG_EPOCHS (15)
DET_EPOCHS = None      # None keeps serpentguard.frontend.DET_EPOCHS (20)
LIMIT = None           # first N photos of each set, for a smoke run

In [2]:
import os
import subprocess
import sys

ALBUMENTATIONS_VERSION = "2.0.8"  # requirements-m3.txt; the per-transform augmentation seeding was verified on it
TIMM_VERSION = "1.0.31"            # requirements-m3.txt; the encoders' feature maps were measured on it
ULTRALYTICS_VERSION = "8.4.202"    # requirements-m3.txt; YOLO26n-seg, YOLO26n and B5's yolov8n-oiv7 run through it
SMP_VERSION = "0.6.0"              # requirements-m3.txt; segmentation-models-pytorch, the U-Net
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"albumentations=={ALBUMENTATIONS_VERSION}",
                f"timm=={TIMM_VERSION}", f"ultralytics=={ULTRALYTICS_VERSION}",
                f"segmentation-models-pytorch=={SMP_VERSION}"], check=True)
os.environ["NO_ALBUMENTATIONS_UPDATE"] = "1"  # otherwise albumentations asks PyPI for its latest version on import

import albumentations
import segmentation_models_pytorch
import timm
import torchvision
import ultralytics

assert albumentations.__version__ == ALBUMENTATIONS_VERSION, f"albumentations {albumentations.__version__} is loaded"
assert timm.__version__ == TIMM_VERSION, f"timm {timm.__version__} is loaded"
assert ultralytics.__version__ == ULTRALYTICS_VERSION, f"ultralytics {ultralytics.__version__} is loaded"
assert segmentation_models_pytorch.__version__ == SMP_VERSION, f"smp {segmentation_models_pytorch.__version__} is loaded"
VERSIONS = {"albumentations": albumentations.__version__, "timm": timm.__version__,
            "ultralytics": ultralytics.__version__, "segmentation_models_pytorch": segmentation_models_pytorch.__version__,
            "torchvision": torchvision.__version__}  # torchvision is Kaggle's, recorded (LR-ASPP comes from it)
print(VERSIONS)

Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
{'albumentations': '2.0.8', 'timm': '1.0.31', 'ultralytics': '8.4.202', 'segmentation_models_pytorch': '0.6.0', 'torchvision': '0.26.0+cu130'}


In [3]:
SCIKIT_LEARN_VERSION = "1.9.1"  # requirements-m3.txt; every metric uses it
subprocess.run([sys.executable, "-m", "pip", "install", "-q", f"scikit-learn=={SCIKIT_LEARN_VERSION}"], check=True)

import sklearn

assert sklearn.__version__ == SCIKIT_LEARN_VERSION, (f"scikit-learn {sklearn.__version__} is loaded; restart the "
                                                     f"kernel after the install")
print(f"scikit-learn {sklearn.__version__}")

scikit-learn 1.9.1


In [4]:
import json
import time
from pathlib import Path

import kagglehub

CODE = Path("/kaggle/input/serpentguard-code/src")
CODE = CODE if CODE.exists() else Path("../src").resolve()
sys.path.insert(0, str(CODE))

import numpy as np
import pandas as pd
import torch

from serpentguard.frontend import (D1, DET_WEIGHTS, FINE_TUNED, MASKS_PRED, PREDS_FILE, S1, SUMMARY_FILE,
                                   TRAIN_CHECK_FILE, WINNERS_FILE, YOLO_WEIGHTS)
from serpentguard.segmenters import YOLO_SEG_WEIGHTS

OWNER = "puneetbajajiitm"


def input_root(slug):
    """Where an attached dataset is mounted; Kaggle no longer mounts datasets at /kaggle/input/<slug>."""
    return Path(kagglehub.dataset_download(f"{OWNER}/{slug}"))


DATA = input_root("serpentguard-indian-snakes-clean")
MASKS = input_root("serpentguard-sam-masks")
SUBSET = input_root("serpentguard-m3-subset") / "subset.csv"
EXTENDED = input_root("serpentguard-extended-sources")
FRONTENDS = Path("/kaggle/working/frontends")
PREDS = Path("/kaggle/working/frontend_preds")
WEIGHTS = Path("/kaggle/tmp/frontend_weights")  # Ultralytics downloads here, outside the saved output
EXPORTS = Path("/kaggle/tmp/frontend_exports")  # the YOLO training sets: derived photos, kept out of the output
POLL_SECONDS = 60
EXPECTED_SUBSET_HASH = "40196d19a68e"  # first 12 hex digits of the real subset's SHA-256

for required in (DATA / "metadata.csv", DATA / "classes.csv", DATA / "data", MASKS / "mask_stats.csv", MASKS / "masks",
                 SUBSET, EXTENDED / "open_images_snake_masks.csv", EXTENDED / "open_images_snake_boxes.csv"):
    assert required.exists(), f"missing input {required}"
gpus = torch.cuda.device_count()
assert gpus >= 1, "no GPU in this session"
workers = max(1, (os.cpu_count() or 2) // min(2, gpus))
FRONTENDS.mkdir(parents=True, exist_ok=True)
WEIGHTS.mkdir(parents=True, exist_ok=True)
# Every job runs in WEIGHTS, so downloading the weights here once spares parallel jobs from downloading them together
# (YOLO's AMP check loads yolo26n.pt in each training process).
subprocess.run([sys.executable, "-c", "import sys; from ultralytics import YOLO; [YOLO(name) for name in sys.argv[1:]]",
                YOLO_SEG_WEIGHTS, DET_WEIGHTS, YOLO_WEIGHTS], cwd=WEIGHTS, check=True)
ENV = {**os.environ, "PYTHONPATH": os.pathsep.join(filter(None, [str(CODE), os.environ.get("PYTHONPATH")]))}
INPUTS = ["--data-root", str(DATA), "--masks-root", str(MASKS), "--subset", str(SUBSET), "--device", "cuda:0",
          "--num-workers", str(workers)] + ([] if LIMIT is None else ["--limit", str(LIMIT)])
print(f"code {CODE}; {gpus} GPUs, {os.cpu_count()} CPUs, {workers} loader workers per job; weights "
      f"{sorted(path.name for path in WEIGHTS.glob('*.pt'))}")


def frontend_command(*args):
    return [sys.executable, "-u", "-m", "serpentguard.frontend", *args, *INPUTS]


def last_line(path):
    lines = path.read_text(errors="replace").strip().splitlines() if path.exists() else []
    return lines[-1][-200:] if lines else "(no output yet)"


def run_on_gpu(name, command, log_path, gpu):
    log_path.parent.mkdir(parents=True, exist_ok=True)
    log = open(log_path, "w")
    process = subprocess.Popen(command, stdout=log, stderr=subprocess.STDOUT, cwd=WEIGHTS,
                               env={**ENV, "CUDA_VISIBLE_DEVICES": str(gpu)})
    print(f"{name} on GPU {gpu}: {' '.join(command)}")
    return process, log


def wait_for(name, process, log, log_path):
    started = time.time()
    while process.poll() is None:
        time.sleep(POLL_SECONDS)
        print(f"{(time.time() - started) / 60:5.1f} min  {name}: {last_line(log_path)}")
    log.close()
    if process.returncode:
        print(log_path.read_text(errors="replace")[-5000:])
    assert process.returncode == 0, f"{name} exited with {process.returncode}; its log tail is above"

code /kaggle/working/repo/src; 2 GPUs, 4 CPUs, 2 loader workers per job; weights ['yolo26n-seg.pt', 'yolo26n.pt', 'yolov8n-oiv7.pt']


## 1. Train the segmenters and the detector

Five jobs: the four S1 segmenters (mask head on MobileNetV4-S, LR-ASPP MobileNetV3, U-Net MobileNetV2, YOLO26n-seg)
and the D1 detector (YOLO26n fine-tuned on the Grounding DINO boxes). The segmenters train on the subset's photos with
valid SAM masks, all with the same photos, epochs, 320 px input and light augmentation. The augmentation matches as
far as the libraries allow: the three torch segmenters get notebook 06's tier 1 pipeline with its camera-style steps
(JPEG compression, downscale, blur, RGB shift), while YOLO26n-seg gets Ultralytics' own flips, affine and HSV settings
at the same strength, without those steps.

Each job is its own `python -m serpentguard.frontend` process, at most one per GPU; the next job starts on whichever
GPU frees first. A job reads its photos once in parallel (the dataset mount is slow on cold files), then writes
`run.json` (photos, epochs, parameters, loss per epoch, warm-up, versions, subset hash), its weights and `train.log`
into its folder. The YOLO training sets go to `/kaggle/tmp/frontend_exports`, so no derived photo lands in the saved
output.

In [5]:
jobs = [(f"{S1}/{kind}", ["train-seg", "--kind", kind, "--out", str(FRONTENDS / S1 / kind),
                          "--export-root", str(EXPORTS)]
         + ([] if SEG_EPOCHS is None else ["--epochs", str(SEG_EPOCHS)])) for kind in SEG_KINDS]
if TRAIN_DETECTOR:
    jobs.append((f"{D1}/{FINE_TUNED}", ["train-det", "--out", str(FRONTENDS / D1 / FINE_TUNED),
                                       "--export-root", str(EXPORTS)]
                 + ([] if DET_EPOCHS is None else ["--epochs", str(DET_EPOCHS)])))
waiting, running, failed, started = list(jobs), {}, [], time.time()
while waiting or running:
    for gpu in [gpu for gpu in range(min(2, gpus)) if gpu not in running]:
        if waiting:
            name, args = waiting.pop(0)
            log_path = FRONTENDS / name / "train.log"
            running[gpu] = (name, log_path, *run_on_gpu(name, frontend_command(*args), log_path, gpu))
    time.sleep(POLL_SECONDS)
    for gpu, (name, log_path, process, log) in list(running.items()):
        print(f"{(time.time() - started) / 60:5.1f} min  GPU {gpu} {name}: {last_line(log_path)}")
        if process.poll() is not None:
            log.close()
            del running[gpu]
            print(f"{name} exited with {process.returncode}")
            if process.returncode:
                failed.append(name)
                print(log_path.read_text(errors="replace")[-5000:])
assert not failed, f"jobs failed: {failed}; their log tails are above"

S1/maskhead on GPU 0: /usr/bin/python3 -u -m serpentguard.frontend train-seg --kind maskhead --out /kaggle/working/frontends/S1/maskhead --export-root /kaggle/tmp/frontend_exports --data-root /kaggle/input/datasets/puneetbajajiitm/serpentguard-indian-snakes-clean --masks-root /kaggle/input/datasets/puneetbajajiitm/serpentguard-sam-masks --subset /kaggle/input/datasets/puneetbajajiitm/serpentguard-m3-subset/subset.csv --device cuda:0 --num-workers 2
S1/lraspp on GPU 1: /usr/bin/python3 -u -m serpentguard.frontend train-seg --kind lraspp --out /kaggle/working/frontends/S1/lraspp --export-root /kaggle/tmp/frontend_exports --data-root /kaggle/input/datasets/puneetbajajiitm/serpentguard-indian-snakes-clean --masks-root /kaggle/input/datasets/puneetbajajiitm/serpentguard-sam-masks --subset /kaggle/input/datasets/puneetbajajiitm/serpentguard-m3-subset/subset.csv --device cuda:0 --num-workers 2
  1.0 min  GPU 0 S1/maskhead: (no output yet)
  1.0 min  GPU 1 S1/lraspp: (no output yet)
  2.0 min 

## 2. Score every candidate

One process on GPU 0. Photos are read the way the predictions read them (exposure fix, no EXIF turn).

- **Segmenters:** mean per-photo IoU and Dice, raw and cleaned. The cleaned score runs the prediction through
  `clean_mask` first and counts a None as an empty mask.
  - Open Images: all 284 photos with human masks (every mask of a photo joined, each resized to the photo). The three
    photos tagged as rotated are counted, since their masks have the stored photo's aspect ratio, and any photo whose
    masks fit the photo turned is listed in `orientation_mismatch`.
  - A 1,000-photo SAM-mask holdout from the calibration set (pseudo-labels).
- **Detectors:** `yolov8n-oiv7.pt` (the B5 model) and the fine-tuned YOLO26n, by precision and recall at confidence
  0.25 and mAP@0.5 over every prediction, on the same 284 photos.

Results go to `frontends/{S1,D1}/<candidate>/final_metrics.json`.

In [6]:
score_log = FRONTENDS / "score.log"
process, log = run_on_gpu("score", frontend_command("score", "--extended-root", str(EXTENDED),
                                                    "--frontends", str(FRONTENDS)), score_log, 0)
wait_for("score", process, log, score_log)
print(sorted(str(path.relative_to(FRONTENDS)) for path in FRONTENDS.glob("*/*/final_metrics.json")))

score on GPU 0: /usr/bin/python3 -u -m serpentguard.frontend score --extended-root /kaggle/input/datasets/puneetbajajiitm/serpentguard-extended-sources --frontends /kaggle/working/frontends --data-root /kaggle/input/datasets/puneetbajajiitm/serpentguard-indian-snakes-clean --masks-root /kaggle/input/datasets/puneetbajajiitm/serpentguard-sam-masks --subset /kaggle/input/datasets/puneetbajajiitm/serpentguard-m3-subset/subset.csv --device cuda:0 --num-workers 2
  1.0 min  score: (no output yet)
  2.0 min  score: (no output yet)
  3.0 min  score: (no output yet)
  4.0 min  score: }
['D1/yolo26n-ft/final_metrics.json', 'D1/yolov8n-oiv7/final_metrics.json', 'S1/lraspp/final_metrics.json', 'S1/maskhead/final_metrics.json', 'S1/unet/final_metrics.json', 'S1/yolo26n-seg/final_metrics.json']


## 3. The winners

`score` writes `winners.json`. The S1 winner has the best Open Images IoU on cleaned masks (what the F-runs read), the
smaller model winning within 0.005, with the raw IoU reported beside it; the D1 winner has the best Open Images
mAP@0.5. This cell checks each winner against that rule on the saved scores.

In [7]:
from serpentguard.frontend_scores import WINNER_TIE, pick_winner

winners = json.loads((FRONTENDS / WINNERS_FILE).read_text())
finals = {stage: {path.parent.name: json.loads(path.read_text())
                  for path in sorted((FRONTENDS / stage).glob("*/final_metrics.json"))} for stage in (S1, D1)}
assert winners[S1]["candidate"] == pick_winner({name: (final["open_images"]["iou_cleaned"], final["parameters"])
                                                for name, final in finals[S1].items()}, tie=WINNER_TIE)
assert winners[D1]["candidate"] == pick_winner({name: (final["detection"]["map50"], final["parameters"])
                                                for name, final in finals[D1].items()}, tie=0.0)
for subset_hash in {final["run"]["subset_hash"] for stage in finals.values() for final in stage.values()}:
    assert subset_hash.startswith(EXPECTED_SUBSET_HASH), f"another subset was read: {subset_hash}"
print(json.dumps({stage: {key: value for key, value in winner.items() if key != "scores"}
                  for stage, winner in winners.items()}, indent=1))

{
 "S1": {
  "candidate": "unet",
  "kind": "unet",
  "weights": "S1/unet/segmenter.pt",
  "rule": "best Open Images IoU of the cleaned masks; within 0.005 the fewest parameters",
  "raw_iou": {
   "lraspp": 0.7905054575932055,
   "maskhead": 0.7725553350329415,
   "unet": 0.8376193317347395,
   "yolo26n-seg": 0.8278211268532348
  }
 },
 "D1": {
  "candidate": "yolo26n-ft",
  "weights": "D1/yolo26n-ft/detector.pt",
  "rule": "best Open Images mAP@0.5"
 }
}


## 4. Predict with the winners

`frontend_preds.csv` gets one row per photo of the subset, full validation, the calibration set and unknown-pool
half A, with empty fields where the winner finds nothing.

- **Boxes:** the detector's best box at confidence 0.05 or above (`BOX_FLOOR`), with that confidence in `box_conf`.
  The decision layer later fits the app's "no snake found" threshold, `tau_snake`, on `box_conf`. `summary.json`
  gives box coverage at the floor and at 0.25 for each set of photos.
- **Masks:** saved at each photo's own resolution, only where `load_mask`'s cleaning keeps them.

`predict` then scores the S1 winner on 1,000 seeded subset photos, its own training photos, against their SAM masks
(`S1/<winner>/train_check.json`). Next to the calibration holdout, that gives the train/validation mask-quality gap
the F-runs inherit. The check below reads a sample back the way the training runner reads it.

In [8]:
predict_log = FRONTENDS / "predict.log"
process, log = run_on_gpu("predict", frontend_command("predict", "--frontends", str(FRONTENDS), "--out", str(PREDS)),
                          predict_log, 0)
wait_for("predict", process, log, predict_log)
summary = json.loads((PREDS / SUMMARY_FILE).read_text())
print(json.dumps(summary["coverage"], indent=1))

predict on GPU 0: /usr/bin/python3 -u -m serpentguard.frontend predict --frontends /kaggle/working/frontends --out /kaggle/working/frontend_preds --data-root /kaggle/input/datasets/puneetbajajiitm/serpentguard-indian-snakes-clean --masks-root /kaggle/input/datasets/puneetbajajiitm/serpentguard-sam-masks --subset /kaggle/input/datasets/puneetbajajiitm/serpentguard-m3-subset/subset.csv --device cuda:0 --num-workers 2
  1.0 min  predict: (no output yet)
  2.0 min  predict: (no output yet)
  3.0 min  predict: (no output yet)
  4.0 min  predict: (no output yet)
  5.0 min  predict: (no output yet)
  6.0 min  predict: (no output yet)
  7.0 min  predict: (no output yet)
  8.0 min  predict: (no output yet)
  9.0 min  predict: (no output yet)
 10.0 min  predict: (no output yet)
 11.0 min  predict: }
{
 "train": {
  "photos": 13925,
  "box": 0.9997845601436266,
  "box_at_conf": 0.9281867145421903,
  "mask": 0.931705565529623
 },
 "val": {
  "photos": 17151,
  "box": 0.9995335548947584,
  "box_at_

In [9]:
from serpentguard.data import build_frame, load_mask, load_rgb, parse_box
from serpentguard.frontend import BOX_FLOOR, prediction_frames, read_inputs

VERIFY_SAMPLE = 20
tables = read_inputs(DATA, None, SUBSET)
frames = prediction_frames(tables["metadata"], tables["classes"], tables["subset"], DATA)
if LIMIT is not None:
    frames = {name: frame.head(LIMIT) for name, frame in frames.items()}
preds = pd.read_csv(PREDS / PREDS_FILE, keep_default_na=False)
expected = sorted(path for frame in frames.values() for path in frame["path"])
assert sorted(preds["path"]) == expected and preds["path"].is_unique, "the predictions don't cover the frames"
assert set(preds["mask_file"].str.startswith(f"{MASKS_PRED}/") | preds["mask_file"].eq("")) == {True}
files = pd.concat(frames.values()).set_index("path")["image_file"]
with_mask = preds[preds["mask_file"].ne("")].sample(min(VERIFY_SAMPLE, int(preds["mask_file"].ne("").sum())), random_state=42)
for path, mask_file in zip(with_mask["path"], with_mask["mask_file"]):
    shape = load_rgb(files[path]).shape[:2]
    assert load_mask(str(PREDS / mask_file), shape) is not None, f"{mask_file} doesn't survive load_mask"
    if shape[0] != shape[1]:  # negative control: the wrong shape must be refused, or the size check is broken
        try:
            load_mask(str(PREDS / mask_file), shape[::-1])
        except ValueError:
            pass
        else:
            raise AssertionError(f"load_mask took {mask_file} at the wrong size")
with_box = preds[preds["box"].ne("")].sample(min(VERIFY_SAMPLE, int(preds["box"].ne("").sum())), random_state=42)
for path, box in zip(with_box["path"], with_box["box"]):
    height, width = load_rgb(files[path]).shape[:2]
    x1, y1, x2, y2 = parse_box(box)
    assert 0 <= x1 < x2 <= width + 1 and 0 <= y1 < y2 <= height + 1, f"box {box} outside {path} ({width} x {height})"
assert preds["box"].eq("").equals(preds["box_conf"].eq("")), "a box without its confidence, or the reverse"
assert pd.to_numeric(preds.loc[preds["box"].ne(""), "box_conf"]).ge(BOX_FLOOR).all(), "a box under the floor"
val = build_frame(tables["metadata"], tables["classes"], ["val"], frontend_preds=pd.read_csv(PREDS / PREDS_FILE),
                  mask_source="predicted", preds_root=PREDS)
val = val[val["path"].isin(frames["val"]["path"])]
assert np.isclose(val["mask_valid"].mean(), summary["coverage"]["val"]["mask"]), "val mask coverage disagrees"
assert np.isclose(val["box"].notna().mean(), summary["coverage"]["val"]["box"]), "val box coverage disagrees"
print(f"{len(preds):,} rows cover every frame photo; {len(with_mask)} sampled masks load at photo size, "
      f"{len(with_box)} sampled boxes lie inside their photo; val coverage reads back through build_frame")

39,526 rows cover every frame photo; 20 sampled masks load at photo size, 20 sampled boxes lie inside their photo; val coverage reads back through build_frame


## 5. The S1 and D1 tables

In [10]:
MINUTES = 60
runs = {name: json.loads((FRONTENDS / S1 / name / "run.json").read_text()) for name in finals[S1]}
checks = {name: json.loads(path.read_text()) for name in finals[S1]
          if (path := FRONTENDS / S1 / name / TRAIN_CHECK_FILE).exists()}  # the winner's, after predict
s1 = pd.DataFrame([{
    "segmenter": name, "parameters (M)": final["parameters"] / 1e6,
    "OI IoU cleaned (picks)": final["open_images"]["iou_cleaned"], "OI IoU raw": final["open_images"]["iou"],
    "OI Dice cleaned": final["open_images"]["dice_cleaned"], "OI Dice raw": final["open_images"]["dice"],
    "OI photos": final["open_images"]["images"], "rotation-tagged (scored)": final["open_images"]["rotation_tagged"],
    "orientation mismatches": len(final["open_images"]["orientation_mismatch"]),
    "holdout IoU cleaned (pseudo)": final["mask_holdout"]["iou_cleaned"],
    "holdout IoU raw (pseudo)": final["mask_holdout"]["iou"], "holdout photos": final["mask_holdout"]["images"],
    "train-photo IoU cleaned (pseudo)": checks[name]["iou_cleaned"] if name in checks else np.nan,
    "train-photo IoU raw (pseudo)": checks[name]["iou"] if name in checks else np.nan,
    "YOLO photos with an instance (OI)": final["open_images"].get("yolo_instances", {}).get("photos_with_instance"),
    "YOLO top conf p50 / p90 / max (OI)": " / ".join(
        f"{final['open_images']['yolo_instances'][key]:.3f}" for key in ("top_conf_p50", "top_conf_p90", "top_conf_max"))
    if "yolo_instances" in final["open_images"] else "",
    "train photos": runs[name]["train_photos"], "train minutes": runs[name]["seconds"] / MINUTES,
    "winner": name == winners[S1]["candidate"]} for name, final in finals[S1].items()]).set_index("segmenter")
D1_LABELS = {"yolov8n-oiv7": "the B5 model, read as the predictions read photos",
             FINE_TUNED: "YOLO26n fine-tuned on the Grounding DINO boxes"}
d1 = pd.DataFrame([{
    "detector": name, "what": D1_LABELS.get(name, name), "parameters (M)": final["parameters"] / 1e6,
    "imgsz": final["run"]["imgsz"],
    "precision @0.25": final["detection"]["precision"], "recall @0.25": final["detection"]["recall"],
    "mAP@0.5": final["detection"]["map50"], "OI photos": final["detection"]["images"],
    "GT boxes": final["detection"]["gt_boxes"], "winner": name == winners[D1]["candidate"]}
    for name, final in finals[D1].items()]).set_index("detector")
s1.to_csv(FRONTENDS / "S1_table.csv")
d1.to_csv(FRONTENDS / "D1_table.csv")
gap = s1.loc[s1["winner"], ["train-photo IoU cleaned (pseudo)", "holdout IoU cleaned (pseudo)"]].iloc[0]
print("S1: segmenters (Open Images = human masks; holdout and train photos = SAM pseudo-labels)")
print(s1.round(3).to_string())
print("\nD1: detectors on the Open Images snake boxes")
print(d1.round(3).to_string())
print(f"\nS1 winner, cleaned IoU against SAM masks: {gap.iloc[0]:.3f} on its own training photos, {gap.iloc[1]:.3f} "
      f"on the calibration holdout (gap {gap.iloc[0] - gap.iloc[1]:+.3f}), the mask-quality gap the F-runs inherit")
coverage = pd.DataFrame(summary["coverage"]).T.rename(columns={
    "box": f"box at conf >= {summary['box_floor']}", "box_at_conf": f"box at conf >= {summary['box_conf_threshold']}"})
print("\nPrediction coverage (share of photos):")
print(coverage.round(3).to_string())

S1: segmenters (Open Images = human masks; holdout and train photos = SAM pseudo-labels)
             parameters (M)  OI IoU cleaned (picks)  OI IoU raw  OI Dice cleaned  OI Dice raw  OI photos  rotation-tagged (scored)  orientation mismatches  holdout IoU cleaned (pseudo)  holdout IoU raw (pseudo)  holdout photos  train-photo IoU cleaned (pseudo)  train-photo IoU raw (pseudo)  YOLO photos with an instance (OI) YOLO top conf p50 / p90 / max (OI)  train photos  train minutes  winner
segmenter                                                                                                                                                                                                                                                                                                                                                                                                    
lraspp                3.218                   0.787       0.791            0.867        0.870        284           